In [1]:
# =============================================================================
# 0. INSTALLS (uncomment if needed)
# =============================================================================
# !pip install git+https://github.com/huggingface/transformers.git -q
# !pip install -U sentence-transformers datasets accelerate torch tqdm

!pip install git+https://github.com/huggingface/transformers.git -q
!pip install accelerate -q
# !pip install sentence-transformers -q 

# sklearn pandas numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0.dev0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 7

In [2]:
# =============================================================================
# IMPORTS
# =============================================================================
import os, json, time, random, logging
import numpy as np, pandas as pd
from dataclasses import dataclass
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, BertModel, MPNetModel, RobertaModel

# =============================================================================
# CONFIGURATION
# =============================================================================
MODEL_MAP = {
    "BERT":        "bert-base-uncased",
    "SBERT":       "sentence-transformers/all-mpnet-base-v2",
    "LegalBERT":   "nlpaueb/legal-bert-base-uncased",
    "InLegalBERT": "law-ai/InLegalBERT"  # Replace with another variant if needed
}

LABEL_MAP = {"SUPPORT": 0, "ATTACK": 1, "NO_REL": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

@dataclass
class TrainingConfig:
    EMBEDDING_BACKBONE: str = "BERT"       # Choose: BERT/ SBERT / LegalBERT / InLegalBERT
    EPOCHS: int = 5
    BATCH_SIZE: int = 32
    MAX_LEN: int = 256
    LR: float = 2e-5
    N_SPLITS: int = 5
    RANDOM_STATE: int = 42
    TEST_SET_SIZE: float = 0.2
    DROPOUT_1: float = 0.1
    DROPOUT_2: float = 0.4
    OUTPUT_DIR: str = "artifacts"
    LOG_FILE: str = "train_eval.log"
    POS_DATA_FILE: str = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE: str = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    FREEZE_UP_TO_LAYER: int = 10
    USE_CHECKPOINTS: bool = True
    CHECKPOINT_PATH: str = "resume_checkpoint.pt"
    CHECKPOINT_EVERY_N_BATCHES: int = 500
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    def hf_model_name(self):
        name = MODEL_MAP.get(self.EMBEDDING_BACKBONE)
        if not name:
            raise ValueError(f"Unknown model: {self.EMBEDDING_BACKBONE}")
        return name

In [3]:
# =============================================================================
# DATA LOADING
# =============================================================================
def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2: fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2: len(parts) - 2]).strip('"')
        label = parts[-1]
    except Exception:
        return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_and_split_data(config: TrainingConfig):
    rows = []
    for fp in [config.POS_DATA_FILE, config.NEG_DATA_FILE]:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                if parsed := parse_lrec_line(line):
                    rows.append(parsed)
    df = pd.DataFrame(rows)
    df["label_id"] = df["label"].map(LABEL_MAP)
    df = df.dropna(subset=["id", "sent1", "sent2", "label_id"])
    train_df, test_df = train_test_split(df, test_size=config.TEST_SET_SIZE,
                                         random_state=config.RANDOM_STATE,
                                         stratify=df["label_id"])
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

# =============================================================================
# DATASET
# =============================================================================
class PairDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df, self.tokenizer, self.max_len = df, tokenizer, max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        enc = self.tokenizer(r["sent1"], r["sent2"],
                             padding="max_length", truncation=True,
                             max_length=self.max_len, return_tensors="pt")
        return {
            "id": int(r["id"]),
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(r["label_id"]), dtype=torch.long)
        }

# =============================================================================
# MODEL
# =============================================================================
class SentencePairClassifier(nn.Module):
    def __init__(self, config: TrainingConfig, hidden_size=768, head_hidden=384, num_labels=3):
        super().__init__()
        self.config = config
        model_name = config.hf_model_name().lower()
        if "minilm" in model_name: # SBERT (MiniLM L6)
            self.transformer = AutoModel.from_pretrained(model_name)
            hidden_size = 384
        elif "mpnet" in model_name.lower():        # SBERT (mpnet)
            self.transformer = MPNetModel.from_pretrained(model_name)
        elif "bert" in model_name.lower():       # LegalBERT / InLegalBERT
            self.transformer = BertModel.from_pretrained(model_name)
        elif "roberta" in model_name:
            self.transformer = RobertaModel.from_pretrained(model_name)
        else:
            self.transformer = AutoModel.from_pretrained(model_name)
            
        print(f"[INFO] Loaded backbone model: {type(self.transformer).__name__}")

        self.dropout1 = nn.Dropout(config.DROPOUT_1)
        self.hidden = nn.Linear(hidden_size, head_hidden)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(config.DROPOUT_2)
        self.out = nn.Linear(head_hidden, num_labels)
        self._freeze_layers_up_to(config.FREEZE_UP_TO_LAYER)
    
    def _freeze_layers_up_to(self, freeze_upto):
        for p in self.transformer.parameters():
            p.requires_grad = False
        encoder = getattr(self.transformer, "encoder", None)
        if encoder and hasattr(encoder, "layer"):
            n_layers = len(encoder.layer)
            print(f"[INFO] {self.config.EMBEDDING_BACKBONE} has {n_layers} layers.")
            freeze_cut = min(freeze_upto, n_layers)
            for i in range(freeze_cut, n_layers):
                for p in encoder.layer[i].parameters():
                    p.requires_grad = True
            print(f"[INFO] Fine-tuning last {n_layers - freeze_cut} layers ({freeze_cut}-{n_layers-1})")
        if hasattr(self.transformer, "pooler") and self.transformer.pooler:
            for p in self.transformer.pooler.parameters():
                p.requires_grad = True
        for p in self.hidden.parameters(): p.requires_grad = True
        for p in self.out.parameters(): p.requires_grad = True

    def forward(self, input_ids, attention_mask):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = self.dropout1(cls)
        x = self.hidden(x)
        x = self.relu(x)
        x = self.dropout2(x)
        return self.out(x)

# =============================================================================
# TRAINER
# =============================================================================
class Trainer:
    def __init__(self, config, train_df, logger):
        self.cfg, self.logger = config, logger
        self.tokenizer = AutoTokenizer.from_pretrained(config.hf_model_name())
        self.train_df = train_df
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    def _build_model(self):
        return SentencePairClassifier(self.cfg).to(self.cfg.DEVICE)

    # ---------- Checkpoint Helpers ----------
    def _save_checkpoint(self, fold, epoch, model, optimizer, scheduler, batch_idx=None):
        if not self.cfg.USE_CHECKPOINTS:
            return
        ckpt_path = os.path.join(self.cfg.OUTPUT_DIR, self.cfg.CHECKPOINT_PATH)
        state = {
            "fold": fold,
            "epoch": epoch,
            "batch_idx": batch_idx if batch_idx is not None else -1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }
        torch.save(state, ckpt_path)
        note = f"(Fold {fold+1}, Epoch {epoch+1}"
        if batch_idx is not None:
            note += f", Batch {batch_idx+1})"
        else:
            note += ")"
        self.logger.info(f"[Checkpoint] Saved {note}")

    def _load_checkpoint(self):
        ckpt_path = os.path.join(self.cfg.OUTPUT_DIR, self.cfg.CHECKPOINT_PATH)
        if os.path.exists(ckpt_path):
            self.logger.info(f"[Checkpoint] Found at {ckpt_path}. Resuming training.")
            return torch.load(ckpt_path, map_location=self.cfg.DEVICE)
        return None

    # ---------- Training ----------
    def _train_epoch(self, model, dataloader, optimizer, scheduler, criterion, fold, epoch):
        model.train()
        tot_loss, correct, total = 0, 0, 0
        for batch_idx, b in enumerate(tqdm(dataloader, desc=f"Training Fold {fold+1} Epoch {epoch+1}")):
            optimizer.zero_grad()
            inp, att, lbl = b["input_ids"].to(self.cfg.DEVICE), b["attention_mask"].to(self.cfg.DEVICE), b["labels"].to(self.cfg.DEVICE)
            logits = model(inp, att)
            loss = criterion(logits, lbl)
            loss.backward(); optimizer.step(); scheduler.step()
            tot_loss += loss.item()
            correct += (logits.argmax(1) == lbl).sum().item(); total += lbl.size(0)

            # ✅ mid-epoch checkpoint
            if (
                self.cfg.USE_CHECKPOINTS
                and (batch_idx + 1) % self.cfg.CHECKPOINT_EVERY_N_BATCHES == 0
            ):
                self._save_checkpoint(fold, epoch, model, optimizer, scheduler, batch_idx)
        return tot_loss / len(dataloader), correct / total

    def eval_metrics(self, model, dataloader, criterion):
        model.eval(); loss_sum, preds, labels = 0, [], []
        with torch.no_grad():
            for b in tqdm(dataloader, desc="Validating"):
                inp, att, lbl = b["input_ids"].to(self.cfg.DEVICE), b["attention_mask"].to(self.cfg.DEVICE), b["labels"].to(self.cfg.DEVICE)
                logits = model(inp, att)
                loss = criterion(logits, lbl)
                loss_sum += loss.item()
                preds.extend(logits.argmax(1).cpu().numpy()); labels.extend(lbl.cpu().numpy())
        acc = accuracy_score(labels, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
        return loss_sum / len(dataloader), acc, prec, rec, f1

    # ---------- KFold Training ----------
    def run_kfold_cv(self):
        kf = KFold(n_splits=self.cfg.N_SPLITS, shuffle=True, random_state=self.cfg.RANDOM_STATE)
        results = []
        ckpt = self._load_checkpoint()
        start_fold = ckpt["fold"] if ckpt else 0
        start_epoch = ckpt["epoch"] + 1 if ckpt and ckpt["batch_idx"] == -1 else (ckpt["epoch"] if ckpt else 0)

        for fold, (tr_idx, va_idx) in enumerate(kf.split(self.train_df)):
            if fold < start_fold:
                continue

            self.logger.info(f"\n=== Fold {fold+1}/{self.cfg.N_SPLITS} ===")
            tr_df, va_df = self.train_df.iloc[tr_idx], self.train_df.iloc[va_idx]
            tr_dl = DataLoader(PairDataset(tr_df, self.tokenizer, self.cfg.MAX_LEN), batch_size=self.cfg.BATCH_SIZE, shuffle=True)
            va_dl = DataLoader(PairDataset(va_df, self.tokenizer, self.cfg.MAX_LEN), batch_size=self.cfg.BATCH_SIZE)
            model = self._build_model()
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=self.cfg.LR)
            scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(tr_dl)*self.cfg.EPOCHS)
            criterion = nn.CrossEntropyLoss()

            if ckpt and ckpt["fold"] == fold:
                model.load_state_dict(ckpt["model_state_dict"])
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                self.logger.info(f"[Resume] Fold {fold+1}, Epoch {start_epoch}")

            best_fold_f1 = -1.0
            for epoch in range(start_epoch, self.cfg.EPOCHS):
                tr_loss, tr_acc = self._train_epoch(model, tr_dl, optimizer, scheduler, criterion, fold, epoch)
                va_loss, va_acc, va_prec, va_rec, va_f1 = self.eval_metrics(model, va_dl, criterion)
                self.logger.info(f"[Fold {fold+1} | Epoch {epoch+1}] Train Acc={tr_acc:.3f} Val F1={va_f1:.3f}")
                if va_f1 > best_fold_f1:
                    best_fold_f1 = va_f1
                    torch.save(model.state_dict(), os.path.join(self.cfg.OUTPUT_DIR, f"best_model_fold_{fold+1}.pt"))
                    self.logger.info(f"✅ New best fold={fold+1}, F1={va_f1:.3f}")
                self._save_checkpoint(fold, epoch, model, optimizer, scheduler)

            results.append({"fold": fold+1, "val_f1": best_fold_f1})

        best_result = max(results, key=lambda x: x["val_f1"])
        best_fold, best_f1 = best_result["fold"], best_result["val_f1"]
        os.system(f"cp {os.path.join(self.cfg.OUTPUT_DIR, f'best_model_fold_{best_fold}.pt')} {os.path.join(self.cfg.OUTPUT_DIR, 'best_model.pt')}")
        info_path = os.path.join(self.cfg.OUTPUT_DIR, "best_model_info.txt")
        with open(info_path, "w") as f:
            f.write(f"Best fold: {best_fold}\nMacro-F1: {best_f1:.4f}\nModel path: best_model_fold_{best_fold}.pt\n")
        self.logger.info(f"✅ Best model from Fold {best_fold} with Macro-F1={best_f1:.3f}")
        if self.cfg.USE_CHECKPOINTS:
            ckpt_path = os.path.join(self.cfg.OUTPUT_DIR, self.cfg.CHECKPOINT_PATH)
            if os.path.exists(ckpt_path): os.remove(ckpt_path)

    # ---------- Evaluation & Embeddings ----------
    def evaluate_and_embed_all(self, train_df, test_df):
        model_path = os.path.join(self.cfg.OUTPUT_DIR, "best_model.pt")
        if not os.path.exists(model_path):
            self.logger.error("best_model.pt not found! Run training first.")
            return

        model = self._build_model()
        model.load_state_dict(torch.load(model_path, map_location=self.cfg.DEVICE))
        model.eval()

        tokenizer = self.tokenizer
        datasets = {
            "train": PairDataset(train_df, tokenizer, self.cfg.MAX_LEN),
            "test":  PairDataset(test_df, tokenizer, self.cfg.MAX_LEN),
            "combined": PairDataset(pd.concat([train_df, test_df]), tokenizer, self.cfg.MAX_LEN)
        }

        for name, ds in datasets.items():
            dl = DataLoader(ds, batch_size=self.cfg.BATCH_SIZE, shuffle=False)
            y_true, y_pred, vecs, ids = [], [], [], []
            with torch.no_grad():
                for b in tqdm(dl, desc=f"Evaluating {name}"):
                    inp, att = b["input_ids"].to(self.cfg.DEVICE), b["attention_mask"].to(self.cfg.DEVICE)
                    lbl = b["labels"].cpu().numpy().tolist()
                    logits = model(inp, att)
                    preds = logits.argmax(1).cpu().numpy().tolist()
                    cls_vec = model.transformer(inp, attention_mask=att).last_hidden_state[:, 0, :].cpu().numpy()
                    vecs.append(cls_vec); ids.extend(b["id"].numpy().tolist())
                    y_true.extend(lbl); y_pred.extend(preds)

            rep = classification_report(y_true, y_pred, target_names=ID_TO_LABEL.values(), zero_division=0, output_dict=True)
            pd.DataFrame(rep).transpose().to_csv(os.path.join(self.cfg.OUTPUT_DIR, f"{name}_report.csv"))
            np.save(os.path.join(self.cfg.OUTPUT_DIR, f"{name}_embeddings.npy"), np.vstack(vecs))
            pd.DataFrame({"id": ids}).to_csv(os.path.join(self.cfg.OUTPUT_DIR, f"{name}_ids.csv"), index=False)
            self.logger.info(f"[{name}] Accuracy={accuracy_score(y_true, y_pred):.4f} | F1={rep['weighted avg']['f1-score']:.4f}")
        self.logger.info("✅ Saved reports and embeddings for train, test, and combined datasets.")

# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    cfg = TrainingConfig(EMBEDDING_BACKBONE="BERT")  # "LegalBERT" or "BERT" or "SBERT" / "InLegalBERT"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    logger = logging.getLogger("train")
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    fh, sh = logging.FileHandler(os.path.join(cfg.OUTPUT_DIR, cfg.LOG_FILE)), logging.StreamHandler()
    fh.setFormatter(fmt); sh.setFormatter(fmt)
    if not logger.handlers: logger.addHandler(fh); logger.addHandler(sh)

    logger.info(f"Using device: {cfg.DEVICE}")
    train_df, test_df = load_and_split_data(cfg)
    trainer = Trainer(cfg, train_df, logger)

    logger.info("Starting 5-Fold training...")
    trainer.run_kfold_cv()

    logger.info("Generating embeddings and reports from best model...")
    trainer.evaluate_and_embed_all(train_df, test_df)

2025-10-24 19:18:28,036 - INFO - Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

2025-10-24 23:55:51,700 - INFO - [train] Accuracy=0.7763 | F1=0.7802


Evaluating test:   0%|          | 0/254 [00:00<?, ?it/s]

2025-10-24 23:59:57,079 - INFO - [test] Accuracy=0.7460 | F1=0.7508


Evaluating combined:   0%|          | 0/1266 [00:00<?, ?it/s]

2025-10-25 00:20:23,973 - INFO - [combined] Accuracy=0.7702 | F1=0.7743
2025-10-25 00:20:23,974 - INFO - ✅ Saved reports and embeddings for train, test, and combined datasets.
